# Figure S1: continuous rainfall and event-scale antecedent rainfall

This notebook uses the processed GPM IMERG products generated by
`01_rainfall_couple.py`. It creates two independent supplementary figures:

1. Annual panels of the continuous rainfall record, with debris-flow initiation
   times marked by event number.
2. Event-wise antecedent rainfall restricted to the 1 h, 6 h and 24 h windows.

The annual display aggregates the original 30 min rainfall increments to daily
totals only for readability. Event-scale accumulations retain the original
30 min calculation and use the interval `(event time - window, event time]`.

## Setup and source files

In [ ]:
from pathlib import Path

import matplotlib as mpl
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

paper_candidates = [
    Path.cwd(),
    Path.cwd() / "uilt_oneday_all" / "uilt_paper_figure",
    Path.cwd().parent,
]
PAPER_DIR = next(
    (
        candidate.resolve()
        for candidate in paper_candidates
        if (candidate / "Figure_S01_rainfall.ipynb").exists()
    ),
    Path(
        r"D:\Pycharm_Project\Find_precursor_signs\ILLgarben\uilt_oneday_all\uilt_paper_figure"
    ),
)
PROJECT_ROOT = PAPER_DIR.parents[1]
RAIN_ANALYSIS_DIR = (
    PROJECT_ROOT
    / "uilt_oneday_all"
    / "03visual"
    / "rain_coupling"
    / "gpm_rainfall_analysis"
)
CONTINUOUS_CSV = RAIN_ANALYSIS_DIR / "GPM_IMERG_Illgraben_30min_continuous_roll.csv"
ANALYSIS_XLSX = RAIN_ANALYSIS_DIR / "GPM_IMERG_Illgraben_rainfall_analysis.xlsx"
OUT_DIR = PAPER_DIR / "Figure_S01_rainfall_exports"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Continuous rainfall: {CONTINUOUS_CSV}")
print(f"Event rainfall: {ANALYSIS_XLSX}")
print(f"Figure output: {OUT_DIR}")

## Load and validate rainfall data

In [ ]:
required_paths = [CONTINUOUS_CSV, ANALYSIS_XLSX]
missing_paths = [path for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError("Missing rainfall inputs: " + ", ".join(map(str, missing_paths)))

continuous = pd.read_csv(
    CONTINUOUS_CSV,
    usecols=["time_utc", "rain_rate_mm_h", "rain_mm_30min", "is_missing_record"],
)
continuous["time_utc"] = pd.to_datetime(continuous["time_utc"], utc=True, errors="coerce")
continuous = continuous.dropna(subset=["time_utc"]).sort_values("time_utc")
continuous["rain_rate_mm_h"] = pd.to_numeric(
    continuous["rain_rate_mm_h"], errors="coerce"
).clip(lower=0)
continuous["rain_mm_30min"] = pd.to_numeric(
    continuous["rain_mm_30min"], errors="coerce"
).clip(lower=0)
if continuous["is_missing_record"].dtype == object:
    continuous["is_missing_record"] = (
        continuous["is_missing_record"].astype(str).str.lower().eq("true")
    )
else:
    continuous["is_missing_record"] = continuous["is_missing_record"].astype(bool)

event_rainfall = pd.read_excel(ANALYSIS_XLSX, sheet_name="event_rainfall")
event_rainfall["event_time_utc"] = pd.to_datetime(
    event_rainfall["event_time_utc"], utc=True, errors="coerce"
)
event_rainfall["event_number"] = pd.to_numeric(
    event_rainfall["event"].astype(str).str.extract(r"(\d+)$")[0],
    errors="coerce",
)
event_rainfall = event_rainfall.sort_values("event_number").reset_index(drop=True)

required_event_columns = {
    "event", "event_time_utc", "event_number",
    "event_cum_1h_mm", "event_cum_6h_mm", "event_cum_1d_mm",
    "event_missing_1h_count", "event_missing_6h_count", "event_missing_24h_count",
}
missing_event_columns = sorted(required_event_columns.difference(event_rainfall.columns))
if missing_event_columns:
    raise ValueError(f"Missing event-rainfall columns: {missing_event_columns}")

daily_rainfall = (
    continuous.set_index("time_utc")
    .resample("D")
    .agg(
        daily_rain_mm=("rain_mm_30min", "sum"),
        missing_30min_count=("is_missing_record", "sum"),
        available_30min_count=("is_missing_record", lambda values: int((~values).sum())),
    )
    .reset_index()
)
daily_rainfall["year"] = daily_rainfall["time_utc"].dt.year
event_rainfall["year"] = event_rainfall["event_time_utc"].dt.year

coverage_start = continuous["time_utc"].min()
coverage_end = continuous["time_utc"].max()
affected_event_mask = (
    event_rainfall[
        ["event_missing_1h_count", "event_missing_6h_count", "event_missing_24h_count"]
    ].sum(axis=1)
    > 0
)

print(
    f"Continuous records: {len(continuous):,}; coverage: "
    f"{coverage_start:%Y-%m-%d} to {coverage_end:%Y-%m-%d} UTC"
)
print(
    f"Daily records: {len(daily_rainfall):,}; internal days with missing 30-min data: "
    f"{int((daily_rainfall['missing_30min_count'] > 0).sum())}"
)
print(f"Debris-flow events: {len(event_rainfall)}")
print(
    "Events with missing records in the 1/6/24 h windows:",
    event_rainfall.loc[affected_event_mask, "event"].tolist(),
)
display(
    event_rainfall[
        ["event", "event_time_utc", "event_cum_1h_mm", "event_cum_6h_mm", "event_cum_1d_mm"]
    ].head()
)

In [ ]:
# ===================== EDIT COMBINED FIGURE STYLE HERE =====================
FIGSIZE_MM = (183, 122)
add_font = 1
FONT = {
    "family": "Arial",
    "base": 7.2 + add_font,
    "axis_label": 8.0 + add_font,
    "tick": 6.3 + add_font,
    "legend": 6.8 + add_font,
    "event_label": 4.9 + add_font,
    "year": 6.3 + add_font,
    "note": 6.2 + add_font,
    "panel_label": 12.0 + add_font,
}
LAYOUT = {
    "left": 0.075,
    "right": 0.985,
    "bottom": 0.185,
    "top": 0.955,
    "hspace": 0.23,
}
RAIN_COLOR = "#3B79A8"
EVENT_COLOR = "#C4473A"
MISSING_COLOR = "#D9D9D9"
BAR_COLORS = {
    "1 h": "#356CA5",
    "6 h": "#67A9CF",
    "24 h": "#2A9D8F",
}
BAR_WIDTH = 0.25
GRID_ALPHA = 0.18
OUTPUT_FORMATS = ( "png", "tiff")
PNG_DPI = 400
TIFF_DPI = 600
# ============================================================================

mpl.rcParams.update(
    {
        "font.family": "sans-serif",
        "font.sans-serif": [FONT["family"]],
        "font.size": FONT["base"],
        "axes.labelsize": FONT["axis_label"],
        "xtick.labelsize": FONT["tick"],
        "ytick.labelsize": FONT["tick"],
        "legend.fontsize": FONT["legend"],
        "axes.linewidth": 0.75,
        "legend.frameon": False,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "svg.fonttype": "none",
    }
)

event_plot = event_rainfall.copy()
event_plot["label"] = "E" + event_plot["event_number"].astype(int).astype(str)
# Keep E39 displayed as a standard event in the combined panel.

fig_combined, (axis_continuous, axis_events) = plt.subplots(
    2,
    1,
    figsize=(FIGSIZE_MM[0] / 25.4, FIGSIZE_MM[1] / 25.4),
    gridspec_kw={"height_ratios": [1.12, 1.0]},
    facecolor="white",
)
fig_combined.subplots_adjust(**LAYOUT)

# -------------------- a: multi-year continuous rainfall --------------------
axis_continuous.bar(
    daily_rainfall["time_utc"],
    daily_rainfall["daily_rain_mm"],
    width=1.0,
    align="edge",
    color=RAIN_COLOR,
    edgecolor="none",
    alpha=0.88,
    zorder=2,
)

combined_start = coverage_start.floor("D")
combined_end = coverage_end.ceil("D")
axis_continuous.set_xlim(combined_start, combined_end)
axis_continuous.set_ylim(0,max(10.0, np.ceil(float(daily_rainfall["daily_rain_mm"].max()) / 10.0) * 10.0 * 1.10),)
axis_continuous.set_ylim(0,65)
for missing_day in daily_rainfall.loc[
    daily_rainfall["missing_30min_count"].gt(0), "time_utc"
]:
    axis_continuous.axvspan(
        missing_day,
        missing_day + pd.Timedelta(days=1),
        color=MISSING_COLOR,
        alpha=0.78,
        linewidth=0,
        zorder=1,
    )

event_label_height = 0.88
for event_row in event_rainfall.itertuples(index=False):
    axis_continuous.axvline(
        event_row.event_time_utc,
        ymin=0,
        ymax=event_label_height - 0.045,
        color=EVENT_COLOR,
        linewidth=0.52,
        alpha=0.62,
        zorder=3,
    )
    axis_continuous.scatter(
        [event_row.event_time_utc],
        [event_label_height],
        marker="v",
        s=13,
        color=EVENT_COLOR,
        edgecolor="white",
        linewidth=0.30,
        transform=axis_continuous.get_xaxis_transform(),
        clip_on=True,
        zorder=5,
    )
axis_continuous.set_ylabel("Daily rainfall (mm)")
axis_continuous.xaxis.set_major_locator(mdates.YearLocator(month=5, day=1))
axis_continuous.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
axis_continuous.xaxis.set_minor_locator(mdates.MonthLocator())
axis_continuous.tick_params(axis="x", labelbottom=True, which="both")
axis_continuous.grid(axis="y", color="0.72", linewidth=0.45, alpha=GRID_ALPHA, zorder=0)
axis_continuous.spines["top"].set_visible(False)
axis_continuous.spines["right"].set_visible(False)
axis_continuous.text(
    -0.065,
    1.08,
    "a",
    transform=axis_continuous.transAxes,
    fontsize=FONT["panel_label"],
    fontweight="bold",
    ha="left",
    va="top",
)
axis_continuous.legend(
    handles=[
        Patch(facecolor=RAIN_COLOR, edgecolor="none", label="Daily rainfall"),
        Line2D(
            [0], [0], marker="v", color=EVENT_COLOR, linestyle="none",
            markersize=4.8, label="Debris-flow initiation",
        ),
        Patch(facecolor=MISSING_COLOR, edgecolor="none", label="Incomplete record"),
    ],
    loc="upper left",
    ncol=3,
    handlelength=1.15,
    columnspacing=1.0,
)

# -------------------- b: event-scale antecedent rainfall -------------------
x = np.arange(len(event_plot), dtype=float)
rainfall_windows = [("event_cum_1h_mm", "1 h", -BAR_WIDTH),("event_cum_6h_mm", "6 h", 0.0),("event_cum_1d_mm", "24 h", BAR_WIDTH),]
for column, label, offset in rainfall_windows:
    bars = axis_events.bar(
        x + offset,
        event_plot[column],
        width=BAR_WIDTH,
        color=BAR_COLORS[label],
        edgecolor="white",
        linewidth=0.30,
        label=label,
        zorder=3,
    )
axis_events.set_xticks(x)
axis_events.set_xticklabels(event_plot["label"], rotation=90, ha="center", va="top")
axis_events.set_xlim(-0.65, len(event_plot) - 0.35)
axis_events.set_ylabel("Antecedent rainfall (mm)")
axis_events.set_xlabel("Debris-flow event")
axis_events.grid(axis="y", color="0.72", linewidth=0.45, alpha=GRID_ALPHA, zorder=0)
axis_events.spines["top"].set_visible(False)
axis_events.spines["right"].set_visible(False)
axis_events.legend(loc="upper left", ncol=3, handlelength=1.15, columnspacing=1.1)

year_values = event_plot["year"].astype(int).to_numpy()
year_change_indices = np.flatnonzero(year_values[1:] != year_values[:-1]) + 1
for boundary in year_change_indices:
    axis_events.axvline(boundary - 0.5, color="0.65", linewidth=0.55, linestyle=":", zorder=1)
group_starts = np.r_[0, year_change_indices]
group_ends = np.r_[year_change_indices, len(event_plot)]
for group_start, group_end in zip(group_starts, group_ends):
    group_center = (group_start + group_end - 1) / 2
    axis_events.text(
        group_center,
        1.015,
        str(year_values[group_start]),
        transform=axis_events.get_xaxis_transform(),
        ha="center",
        va="bottom",
        fontsize=FONT["year"],
        color="0.35",
    )

axis_events.text(
    -0.065,
    1.10,
    "b",
    transform=axis_events.transAxes,
    fontsize=FONT["panel_label"],
    fontweight="bold",
    ha="left",
    va="top",
)
combined_stem = OUT_DIR / "FigureS1"
combined_paths = []
for extension in OUTPUT_FORMATS:
    output_path = combined_stem.with_suffix(f".{extension}")
    save_options = {
        "bbox_inches": "tight",
        "facecolor": "white",
        "pad_inches": 0.04,
    }
    if extension == "png":
        save_options["dpi"] = PNG_DPI
    elif extension == "tiff":
        save_options["dpi"] = TIFF_DPI
        save_options["pil_kwargs"] = {"compression": "tiff_lzw"}
    fig_combined.savefig(output_path, **save_options)
    combined_paths.append(output_path)

display(fig_combined)
plt.close(fig_combined)
print("Combined rainfall exports:", [path.name for path in combined_paths])


In [ ]:
import matplotlib as mpl
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

# ===================== EDIT COMBINED FIGURE STYLE HERE =====================
# JGR/AGU双栏宽度
FIGSIZE_MM = (183, 122)

add_font = 2
FONT = {
    "family": "Arial",
    "base": 7.2 + add_font,
    "axis_label": 8.0 + add_font,
    "tick": 6.3 + add_font,
    "legend": 6.8 + add_font,
    "event_label": 4.9 + add_font,
    "year": 6.3 + add_font,
    "note": 6.2 + add_font,
    "panel_label": 12.0 + add_font,
}
LAYOUT = {"left": 0.075, "right": 0.985, "bottom": 0.185, "top": 0.955, "hspace": 0.29}

# 色觉友好、低饱和度的JGR风格配色
RAIN_COLOR = "#4477AA"  # muted blue
RAIN_COLOR = "tab:blue"  # muted blue
EVENT_COLOR = "#CC6677"  # muted reddish purple
EVENT_COLOR = "tab:red"  # muted reddish purple

MISSING_COLOR = "#D9D9D9"  # neutral grey

BAR_COLORS = {"1 h": "#52ddbb", "6 h": "#ffa17a", "24 h": "#56a2d9"}
BAR_WIDTH = 0.25
CONTINUOUS_YMAX = 65
GRID_COLOR = "#B8B8B8"
GRID_ALPHA = 0.32
GRID_LINEWIDTH = 0.45
AXIS_LINEWIDTH = 0.75
TICK_WIDTH = 0.65
TICK_LENGTH = 3.0
OUTPUT_FORMATS = ("png", "tiff")
PNG_DPI = 400
TIFF_DPI = 600
# ============================================================================

mpl.rcParams.update(
    {
        "font.family": "sans-serif",
        "font.sans-serif": [FONT["family"], "Helvetica", "DejaVu Sans"],
        "font.size": FONT["base"],
        "axes.labelsize": FONT["axis_label"],
        "xtick.labelsize": FONT["tick"],
        "ytick.labelsize": FONT["tick"],
        "legend.fontsize": FONT["legend"],
        "axes.linewidth": AXIS_LINEWIDTH,
        "axes.axisbelow": True,
        "xtick.direction": "out",
        "ytick.direction": "out",
        "xtick.major.width": TICK_WIDTH,
        "ytick.major.width": TICK_WIDTH,
        "xtick.major.size": TICK_LENGTH,
        "ytick.major.size": TICK_LENGTH,
        "legend.frameon": False,
        "figure.facecolor": "white",
        "axes.facecolor": "white",
        "savefig.facecolor": "white",
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "svg.fonttype": "none",
    }
)

# ------------------------- Prepare event labels -----------------------------
event_plot = event_rainfall.copy()
event_plot["label"] = "E" + event_plot["event_number"].astype(int).astype(str)

# ---------------------------- Create figure ---------------------------------
fig_combined, (axis_continuous, axis_events) = plt.subplots(
    nrows=2,
    ncols=1,
    figsize=(FIGSIZE_MM[0] / 25.4, FIGSIZE_MM[1] / 25.4),
    gridspec_kw={"height_ratios": [1.12, 1.0]},
    facecolor="white",
)
fig_combined.subplots_adjust(**LAYOUT)

# ============================================================================
# a. Multi-year continuous rainfall
# ============================================================================
axis_continuous.bar(
    daily_rainfall["time_utc"],
    daily_rainfall["daily_rain_mm"],
    width=1.0,
    align="edge",
    color=RAIN_COLOR,
    edgecolor="none",
    alpha=0.88,
    zorder=2,
)

combined_start = coverage_start.floor("D")
combined_end = coverage_end.ceil("D")
axis_continuous.set_xlim(combined_start, combined_end)
axis_continuous.set_ylim(0, CONTINUOUS_YMAX)
axis_continuous.margins(x=0)

# ------------------------ Incomplete-record intervals -----------------------
for missing_day in daily_rainfall.loc[
    daily_rainfall["missing_30min_count"].gt(0), "time_utc"
]:
    axis_continuous.axvspan(
        missing_day,
        missing_day + pd.Timedelta(days=1),
        color=MISSING_COLOR,
        alpha=0.75,
        linewidth=0,
        zorder=1,
    )

# ------------------------- Debris-flow initiation ---------------------------
event_label_height = 0.88
for event_row in event_rainfall.itertuples(index=False):
    axis_continuous.axvline(
        event_row.event_time_utc,
        ymin=0,
        ymax=event_label_height - 0.045,
        color=EVENT_COLOR,
        linewidth=0.55,
        alpha=0.67,
        zorder=3,
    )
    axis_continuous.scatter(
        [event_row.event_time_utc],
        [event_label_height],
        marker="v",
        s=14,
        facecolor=EVENT_COLOR,
        edgecolor="white",
        linewidth=0.35,
        transform=axis_continuous.get_xaxis_transform(),
        clip_on=True,
        zorder=5,
    )

# ------------------------------- Axes ---------------------------------------
axis_continuous.set_ylabel("Daily rainfall (mm)", labelpad=3)
axis_continuous.set_xlabel("Year", labelpad=3)
axis_continuous.xaxis.set_major_locator(mdates.YearLocator(month=1, day=1))
axis_continuous.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
axis_continuous.xaxis.set_minor_locator(mdates.MonthLocator(interval=3))
axis_continuous.tick_params(axis="x", which="major", labelbottom=True, pad=2)
axis_continuous.tick_params(axis="x", which="minor", length=1.8, width=0.45)
axis_continuous.tick_params(axis="y", pad=2)
axis_continuous.spines["top"].set_visible(False)
axis_continuous.spines["right"].set_visible(False)

# ----------------------------- Panel label ----------------------------------
axis_continuous.text(-0.065,1.08, "a",transform=axis_continuous.transAxes,
    fontsize=FONT["panel_label"],fontweight="bold",ha="left",va="top",)
# -------------------------------- Legend ------------------------------------
axis_continuous.legend(
    handles=[Patch(facecolor=RAIN_COLOR, edgecolor="none", label="Daily rainfall"),
    Line2D([0],[0],marker="v",color=EVENT_COLOR,markerfacecolor=EVENT_COLOR,markeredgecolor="white",markeredgewidth=0.35,linestyle="none",markersize=5.0,label="Debris-flow initiation",),Patch(facecolor=MISSING_COLOR, edgecolor="none", label="Incomplete record"),],
    loc="upper left",ncol=3,handlelength=1.15,handletextpad=0.45,columnspacing=1.0,borderaxespad=0.25,)
# ============================================================================
# b. Event-scale antecedent rainfall
# ============================================================================
x = np.arange(len(event_plot), dtype=float)
rainfall_windows = [("event_cum_1h_mm", "1 h", -BAR_WIDTH),
                    ("event_cum_6h_mm", "6 h", 0.0), ("event_cum_1d_mm", "24 h", BAR_WIDTH),]

for column, label, offset in rainfall_windows:
    axis_events.bar(
        x + offset,
        event_plot[column],
        width=BAR_WIDTH,
        color=BAR_COLORS[label],
        edgecolor="white",
        linewidth=0.30,
        label=label,
        zorder=3,
    )

# ------------------------------- Axes ---------------------------------------
axis_events.set_xticks(x)
axis_events.set_xticklabels(event_plot["label"],rotation=90,  
                    ha="center",va="top", fontsize=FONT["event_label"],)
axis_events.set_xlim(-0.65, len(event_plot) - 0.35)
axis_events.margins(x=0)
axis_events.set_ylabel("Antecedent rainfall (mm)", labelpad=3)
axis_events.set_xlabel("Debris-flow event", labelpad=4)
axis_events.tick_params(axis="x", pad=1.5)
axis_events.tick_params(axis="y", pad=2)
axis_events.spines["top"].set_visible(False)
axis_events.spines["right"].set_visible(False)
# -------------------------------- Legend ------------------------------------
axis_events.legend(loc="upper left",ncol=3,handlelength=1.15,handletextpad=0.45, columnspacing=1.1, borderaxespad=0.25,)
# ---------------------------- Year boundaries -------------------------------
year_values = event_plot["year"].astype(int).to_numpy()
year_change_indices = np.flatnonzero(year_values[1:] != year_values[:-1]) + 1
for boundary in year_change_indices:
    axis_events.axvline(
        boundary - 0.5,
        color="#888888",
        linewidth=0.55,
        linestyle=":",
        alpha=0.75,
        zorder=1,
    )

# ------------------------------ Year labels ---------------------------------
group_starts = np.r_[0, year_change_indices]
group_ends = np.r_[year_change_indices, len(event_plot)]

for group_start, group_end in zip(group_starts, group_ends):
    group_center = (group_start + group_end - 1) / 2
    axis_events.text(group_center, 1.015,str(year_values[group_start]),transform=axis_events.get_xaxis_transform(), ha="center",
        va="bottom",
        fontsize=FONT["year"],
        color="#4D4D4D",
    )

# ----------------------------- Panel label ----------------------------------
axis_events.text(-0.065,1.13,"b",transform=axis_events.transAxes,fontsize=FONT["panel_label"],
                 fontweight="bold",ha="left", va="top",)


# ============================================================================
# Export
# ============================================================================
for ax in (axis_continuous,axis_events):
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_color("black")
        spine.set_linewidth(AXIS_LINEWIDTH)
    ax.tick_params(axis="both",which="both",direction="out",bottom=True,left=True,top=False,right=False,labeltop=False,labelright=False)
from pathlib import Path
save_root=Path(r"D:\Pycharm_Project\Find_precursor_signs\ILLgarben\uilt_oneday_all\uilt_paper_figure\Fig_all")
(save_root/"png").mkdir(parents=True,exist_ok=True)
(save_root/"tiff").mkdir(parents=True,exist_ok=True)
png_path=save_root/"png"/"Fig S1.png"
tiff_path=save_root/"tiff"/"Fig S1.tiff"
fig_combined.savefig(png_path,format="png",dpi=PNG_DPI,bbox_inches="tight",pad_inches=0,facecolor="white",transparent=False)
fig_combined.savefig(tiff_path,format="tiff",dpi=TIFF_DPI,bbox_inches="tight",pad_inches=0,facecolor="white",transparent=False,pil_kwargs={"compression":"tiff_lzw"})
display(fig_combined)
plt.close(fig_combined)
print("Saved Fig S1:")
print(png_path)
print(tiff_path)











combined_stem = OUT_DIR / "FigureS1"
combined_paths = []

for extension in OUTPUT_FORMATS:
    output_path = combined_stem.with_suffix(f".{extension}")
    save_options = {
        "bbox_inches": "tight",
        "facecolor": "white",
        "pad_inches": 0.04,
    }
    if extension == "png":
        save_options["dpi"] = PNG_DPI
    elif extension == "tiff":
        save_options["dpi"] = TIFF_DPI
        save_options["pil_kwargs"] = {"compression": "tiff_lzw"}
    print(output_path)
    fig_combined.savefig(output_path, **save_options)
    combined_paths.append(output_path)

display(fig_combined)
plt.close(fig_combined)
print("Combined rainfall exports:", [path.name for path in combined_paths])